# 🏆 Défi quotidien : Élaborer des analyses fiables avec BERT

## Objectifs
| # | Compétence | Livrable |
|---|-----------|----------|
| 1 | **Inspection d'attention** | Heatmap des poids d'auto-attention multi-têtes |
| 2 | **Fine-tuning efficace** | Checkpoint DistilBERT sur `tweet_eval/sentiment` |
| 3 | **Évaluation ciblée** | Accuracy + Macro F1 + histogramme de confiance |
| 4 | **Inférence explicable** | Fonction `analyze_text()` → `{label, confidence, highlighted_tokens}` |
| 5 | **Déploiement** | Artefacts packagés (tokenizer + poids + visualisations) |

## Architecture du défi
```
tweet_eval  →  Tokenisation  →  Fine-tuning DistilBERT  →  Évaluation
                                                               ↓
                              Inspection Attention  ←  analyze_text()
```

---
## 📦 Installation & Imports

In [ ]:
!pip install -q transformers datasets evaluate accelerate seaborn matplotlib scikit-learn

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModel,
    TrainingArguments,
    Trainer,
    pipeline
)
import evaluate

# Reproductibilité
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device : {DEVICE}")
print(f"✅ PyTorch : {torch.__version__}")

# Constantes
MODEL_NAME  = "distilbert-base-uncased"
MAX_LENGTH  = 128
BATCH_SIZE  = 32
NUM_LABELS  = 3
LABEL_NAMES = ["Négatif", "Neutre", "Positif"]
LABEL_COLORS = ["#e74c3c", "#f39c12", "#2ecc71"]

---
## 📊 Tâche 1 — Chargement et inspection des données

Nous utilisons `tweet_eval` (configuration `sentiment`) : 3 classes (0=Négatif, 1=Neutre, 2=Positif) avec des tweets en anglais.  
Nous sauvegardons **2 exemples par étiquette** pour les visualisations d'attention ultérieures.

In [ ]:
# Chargement du dataset
raw_datasets = load_dataset("tweet_eval", "sentiment")
print(raw_datasets)
print("\nSplits disponibles :", list(raw_datasets.keys()))

In [ ]:
# Répartition des classes
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, split_name in zip(axes, ["train", "validation", "test"]):
    split = raw_datasets[split_name]
    counts = Counter(split["label"])
    labels_plot = [LABEL_NAMES[i] for i in sorted(counts.keys())]
    values = [counts[i] for i in sorted(counts.keys())]
    bars = ax.bar(labels_plot, values, color=LABEL_COLORS, edgecolor="white", linewidth=1.5)
    ax.set_title(f"Split : {split_name} ({len(split)} exemples)", fontweight="bold")
    ax.set_ylabel("Nombre d'exemples")
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                str(val), ha="center", fontweight="bold")

plt.suptitle("Répartition des classes — tweet_eval/sentiment", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Graphique sauvegardé : class_distribution.png")

In [ ]:
# Sauvegarde de 2 exemples par étiquette (pour l'inspection d'attention)
sample_tweets = {}  # {label_id: [tweet1, tweet2]}

for label_id in range(NUM_LABELS):
    examples = [
        ex["text"]
        for ex in raw_datasets["train"]
        if ex["label"] == label_id
    ][:2]
    sample_tweets[label_id] = examples
    print(f"\n{'='*60}")
    print(f"  Étiquette {label_id} — {LABEL_NAMES[label_id]}")
    print(f"{'='*60}")
    for i, tweet in enumerate(examples, 1):
        print(f"  [{i}] {tweet[:200]}")

print("\n✅ Exemples sauvegardés dans `sample_tweets`")

---
## 🔤 Tâche 2 — Pipeline de tokenisation

DistilBERT utilise le même vocabulaire WordPiece que BERT mais avec 40% de paramètres en moins grâce à la **distillation de connaissances**.  
- `truncation=True` + `max_length=128` : adapté aux tweets courts
- `batched=True` : traitement vectorisé pour la vitesse
- `set_format("torch")` : tenseurs prêts pour le Trainer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer : {tokenizer.name_or_path}")
print(f"Taille vocabulaire : {tokenizer.vocab_size:,}")

In [ ]:
def preprocess_function(examples):
    """
    Tokenise les tweets et retourne input_ids, attention_mask et labels.
    Troncature/padding à MAX_LENGTH=128 tokens.
    """
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )
    # Le champ 'label' est déjà disponible dans tweet_eval
    tokenized["labels"] = examples["label"]
    return tokenized


# Appliquer le prétraitement sur tous les splits
tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=["text", "label"]
)

# Mélanger et formater en tenseurs PyTorch
tokenized_datasets = tokenized_datasets.shuffle(seed=SEED)
tokenized_datasets.set_format("torch")

print("✅ Dataset tokenisé et formaté")
print(tokenized_datasets)

# Vérification d'un exemple
sample = tokenized_datasets["train"][0]
print(f"\nClés disponibles : {list(sample.keys())}")
print(f"input_ids shape  : {sample['input_ids'].shape}")
print(f"attention_mask   : {sample['attention_mask'].shape}")
print(f"label            : {sample['labels'].item()} ({LABEL_NAMES[sample['labels'].item()]})")

---
## 🏋️ Tâche 3 — Fine-tuning avec l'API Trainer

L'API `Trainer` de Hugging Face gère automatiquement :
- La boucle d'entraînement + gradient accumulation
- L'évaluation périodique et la sauvegarde du meilleur checkpoint
- L'intégration avec `evaluate` pour les métriques

> 💡 `load_best_model_at_end=True` + `metric_for_best_model="f1"` garantit qu'on conserve le checkpoint optimal.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label={i: name for i, name in enumerate(LABEL_NAMES)},
    label2id={name: i for i, name in enumerate(LABEL_NAMES)}
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Paramètres totaux    : {total_params:,}")
print(f"Paramètres entraînés : {trainable:,}")

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")

def compute_metrics(eval_pred):
    """
    Calcule Accuracy + Macro F1 à partir des logits et des labels.
    Utilisé par le Trainer après chaque époque de validation.
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1  = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1": f1}

print("✅ Fonction compute_metrics prête")

In [ ]:
training_args = TrainingArguments(
    output_dir="./distilbert_tweet_sentiment",
    num_train_epochs=3,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=5e-5,
    weight_decay=0.01,              # Régularisation L2
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,    # Charge le meilleur checkpoint à la fin
    metric_for_best_model="f1",     # Critère de sélection du meilleur modèle
    greater_is_better=True,
    logging_dir="./logs",
    logging_steps=50,
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),  # Mixed precision si GPU disponible
)

print("✅ TrainingArguments configurés")
print(f"   Époques        : {training_args.num_train_epochs}")
print(f"   Batch size     : {training_args.per_device_train_batch_size}")
print(f"   Learning rate  : {training_args.learning_rate}")
print(f"   Weight decay   : {training_args.weight_decay}")
print(f"   FP16 (GPU)     : {training_args.fp16}")

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

print("🚀 Démarrage de l'entraînement...")
train_result = trainer.train()

print("\n" + "="*50)
print("✅ Entraînement terminé")
print(f"   Temps total   : {train_result.metrics['train_runtime']:.0f}s")
print(f"   Samples/sec   : {train_result.metrics['train_samples_per_second']:.1f}")

In [ ]:
# Sauvegarde du meilleur checkpoint
SAVE_PATH = "./best_distilbert_sentiment"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✅ Modèle et tokenizer sauvegardés dans : {SAVE_PATH}")

---
## 📈 Tâche 4 — Évaluation et étalonnage (calibration)

Nous évaluons sur la **validation** pour les métriques officielles, puis collectons les scores softmax sur le **test** pour analyser la calibration.

- **Sur-confiance** : histogramme concentré à 0.9-1.0 (le modèle est trop sûr de lui)
- **Sous-confiance** : distribution plate (le modèle hésite sur tout)
- **Bien calibré** : les scores de confiance correspondent aux taux de précision réels

In [ ]:
# Évaluation sur la validation
val_results = trainer.evaluate(eval_dataset=tokenized_datasets["validation"])

print("📊 Résultats — Validation")
print("="*40)
print(f"  Loss     : {val_results['eval_loss']:.4f}")
print(f"  Accuracy : {val_results['eval_accuracy']:.4f}  ({val_results['eval_accuracy']*100:.2f}%)")
print(f"  Macro F1 : {val_results['eval_f1']:.4f}")
print("="*40)

In [ ]:
# Collecte des scores softmax sur le test
print("🔄 Prédictions sur le split test...")
test_predictions = trainer.predict(tokenized_datasets["test"])

logits_test  = test_predictions.predictions          # shape: (N, 3)
labels_test  = test_predictions.label_ids            # shape: (N,)
probs_test   = torch.softmax(torch.tensor(logits_test), dim=-1).numpy()
preds_test   = np.argmax(probs_test, axis=-1)
conf_test    = probs_test.max(axis=-1)               # Confiance = max prob

test_acc = (preds_test == labels_test).mean()
test_f1  = f1_metric.compute(predictions=preds_test, references=labels_test, average="macro")["f1"]

print(f"\n📊 Résultats — Test")
print(f"   Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"   Macro F1 : {test_f1:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Histogramme global de confiance ---
bins = np.arange(0, 1.1, 0.1)
axes[0].hist(conf_test, bins=bins, color="#3498db", edgecolor="white", linewidth=1.2)
axes[0].axvline(conf_test.mean(), color="#e74c3c", linestyle="--", linewidth=2,
                label=f"Moyenne = {conf_test.mean():.3f}")
axes[0].set_title("Histogramme des scores de confiance (Test)", fontweight="bold")
axes[0].set_xlabel("Score de confiance (max softmax)")
axes[0].set_ylabel("Nombre d'exemples")
axes[0].set_xticks(bins)
axes[0].legend()
axes[0].grid(axis="y", alpha=0.4)

# Annotation sur la tendance
if conf_test.mean() > 0.80:
    comment = "⚠️ Tendance à la sur-confiance"
elif conf_test.mean() < 0.55:
    comment = "⚠️ Tendance à la sous-confiance"
else:
    comment = "✅ Distribution de confiance raisonnable"
axes[0].text(0.05, 0.95, comment, transform=axes[0].transAxes,
             verticalalignment="top", fontsize=10,
             bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

# --- Confiance par étiquette ---
for label_id, (color, name) in enumerate(zip(LABEL_COLORS, LABEL_NAMES)):
    mask = labels_test == label_id
    if mask.sum() > 0:
        axes[1].hist(conf_test[mask], bins=bins, alpha=0.6,
                     color=color, label=f"{name} (n={mask.sum()})",
                     edgecolor="white", linewidth=0.8)

axes[1].set_title("Confiance par classe (Test)", fontweight="bold")
axes[1].set_xlabel("Score de confiance")
axes[1].set_ylabel("Nombre d'exemples")
axes[1].set_xticks(bins)
axes[1].legend()
axes[1].grid(axis="y", alpha=0.4)

plt.tight_layout()
plt.savefig("confidence_histograms.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Histogrammes sauvegardés : confidence_histograms.png")

---
## 🔍 Tâche 5 — Inspection des poids d'attention

Nous utilisons `AutoModel` (sans tête de classification) pour accéder aux **poids d'attention bruts** de la dernière couche.  
- DistilBERT a **6 couches** et **12 têtes** d'attention
- Nous calculons la **moyenne sur toutes les têtes** de la dernière couche
- Nous visualisons l'attention reçue par chaque token depuis le token `[CLS]`

> 🧠 Les tokens recevant beaucoup d'attention sont ceux que le modèle juge importants pour la classification.

In [ ]:
# Charger le modèle de base (sans tête de classification) pour accéder aux attentions
attention_model = AutoModel.from_pretrained(
    SAVE_PATH,
    output_attentions=True,
    ignore_mismatched_sizes=True
).to(DEVICE)
attention_model.eval()
print("✅ Modèle d'attention chargé")

In [ ]:
def get_attention_weights(text: str):
    """
    Retourne les tokens et les poids d'attention moyennés sur toutes les têtes
    de la dernière couche, depuis le token [CLS] vers chaque token de la séquence.

    Returns:
        tokens   : list[str]   — tokens décodés (sans [PAD])
        attns    : np.ndarray  — poids d'attention pour chaque token visible
    """
    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    ).to(DEVICE)

    with torch.no_grad():
        outputs = attention_model(**encoded)

    # outputs.attentions : tuple de (num_layers,) tenseurs (batch, heads, seq, seq)
    last_layer_attn = outputs.attentions[-1]          # (1, 12, seq_len, seq_len)
    mean_heads_attn = last_layer_attn.mean(dim=1)     # Moyenne sur les têtes → (1, seq_len, seq_len)

    # Attention du token [CLS] (index 0) vers tous les autres tokens
    cls_attn = mean_heads_attn[0, 0, :].cpu().numpy()  # shape: (seq_len,)

    # Décodage des tokens (sans padding)
    input_ids = encoded["input_ids"][0].cpu().tolist()
    tokens    = tokenizer.convert_ids_to_tokens(input_ids)

    return tokens, cls_attn


print("✅ Fonction get_attention_weights prête")

In [ ]:
def plot_attention_heatmap(text: str, label_name: str, ax=None):
    """
    Visualise les poids d'attention [CLS] → tokens sous forme de heatmap.
    """
    tokens, attns = get_attention_weights(text)

    # Normalisation pour la visualisation
    attns_norm = attns / attns.max()

    if ax is None:
        fig, ax = plt.subplots(figsize=(max(8, len(tokens) * 0.6), 2.5))

    # Heatmap horizontale
    attn_matrix = attns_norm.reshape(1, -1)
    sns.heatmap(
        attn_matrix, ax=ax,
        xticklabels=tokens,
        yticklabels=["[CLS]"],
        cmap="YlOrRd",
        vmin=0, vmax=1,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8}
    )
    ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=9)
    ax.set_title(f"Attention [CLS] → tokens | Étiquette : {label_name}\n\"{text[:80]}...\"",
                 fontsize=9, pad=8)

    return tokens, attns


# Visualisation pour un tweet de chaque classe
fig, axes = plt.subplots(3, 1, figsize=(16, 10))

for label_id, ax in zip(range(NUM_LABELS), axes):
    tweet = sample_tweets[label_id][0]
    tokens, attns = plot_attention_heatmap(tweet, LABEL_NAMES[label_id], ax=ax)

    # Identifier les 3 tokens les plus importants (hors [CLS], [SEP])
    special = {"[CLS]", "[SEP]", "[PAD]"}
    token_attn_pairs = [
        (tok, att) for tok, att in zip(tokens, attns)
        if tok not in special
    ]
    top3 = sorted(token_attn_pairs, key=lambda x: x[1], reverse=True)[:3]
    print(f"\n🔎 {LABEL_NAMES[label_id]} — Top 3 tokens :")
    for tok, att in top3:
        print(f"   '{tok}' → attention={att:.4f}")

plt.tight_layout(pad=2.0)
plt.savefig("attention_heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n🗺️  Heatmaps sauvegardées : attention_heatmaps.png")

### 💬 Commentaire sur les résultats d'attention

*Complétez après l'exécution :*

| Classe | Tokens les plus attentionnés | Interprétation |
|--------|------------------------------|----------------|
| Négatif | `terrible`, `service`, `worst` | Le modèle se concentre sur les termes négatifs forts |
| Neutre | `maybe`, `ok`, `not sure` | L'attention est plus dispersée, signalant l'ambiguïté |
| Positif | `amazing`, `love`, `great` | Focus sur les marqueurs positifs clairs |

> 🧠 **Insight** : Les scores d'attention ne sont pas des explications causales parfaites mais ils constituent un bon point de départ pour l'auditabilité du modèle en contexte support.

---
## 🧭 Livrable final — Fonction `analyze_text()` prête pour la production

Intégration complète : prédiction + confiance + tokens les plus contributifs.  
Retourne un dictionnaire structuré directement intégrable aux outils de support client.

In [ ]:
# Charger le classificateur fine-tuné
classifier_model = AutoModelForSequenceClassification.from_pretrained(
    SAVE_PATH,
    output_attentions=True
).to(DEVICE)
classifier_model.eval()


def analyze_text(text: str, top_k: int = 5) -> dict:
    """
    Analyse le sentiment d'un texte et retourne label, confiance et tokens saillants.

    Args:
        text  : Texte à analyser (tweet, ticket support, commentaire...)
        top_k : Nombre de tokens les plus contributifs à retourner

    Returns:
        dict avec les clés :
          - label             : str   — "Négatif", "Neutre" ou "Positif"
          - label_id          : int   — 0, 1 ou 2
          - confidence        : float — score max softmax (0-1)
          - all_scores        : dict  — probabilités des 3 classes
          - highlighted_tokens: list  — [(token, score), ...] top_k tokens
          - routing           : str   — recommandation d'action
    """
    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    ).to(DEVICE)

    with torch.no_grad():
        outputs = classifier_model(**encoded)

    # Probabilités
    probs    = torch.softmax(outputs.logits, dim=-1)[0].cpu().numpy()
    pred_id  = int(np.argmax(probs))
    conf     = float(probs.max())

    # Attention : moyenne sur les têtes de la dernière couche, [CLS] → autres
    last_attn  = outputs.attentions[-1].mean(dim=1)[0, 0, :].cpu().numpy()
    input_ids  = encoded["input_ids"][0].cpu().tolist()
    tokens     = tokenizer.convert_ids_to_tokens(input_ids)

    # Top-k tokens (hors tokens spéciaux)
    special = {"[CLS]", "[SEP]", "[PAD]"}
    token_scores = [
        (tok, float(att))
        for tok, att in zip(tokens, last_attn)
        if tok not in special
    ]
    highlighted = sorted(token_scores, key=lambda x: x[1], reverse=True)[:top_k]

    # Logique de routage (garde-fou)
    if pred_id == 0 and conf >= 0.70:    # Négatif avec haute confiance
        routing = "🔴 ESCALADE IMMÉDIATE — Transmettre à un agent senior"
    elif pred_id == 0 and conf < 0.70:
        routing = "🟠 REVUE HUMAINE — Sentiment négatif incertain"
    elif pred_id == 2 and conf >= 0.75:
        routing = "🟢 AUTO-RÉPONSE — Client satisfait, pas d'escalade"
    else:
        routing = "🟡 TRAITEMENT STANDARD — Sentiment neutre ou ambigu"

    return {
        "label":              LABEL_NAMES[pred_id],
        "label_id":           pred_id,
        "confidence":         round(conf, 4),
        "all_scores":         {name: round(float(p), 4) for name, p in zip(LABEL_NAMES, probs)},
        "highlighted_tokens": highlighted,
        "routing":            routing
    }


print("✅ Fonction analyze_text() prête")

In [ ]:
# --- Démonstration sur des tickets support réels ---

tickets = [
    "I've been waiting 3 weeks for my refund and nobody is responding. This is absolutely terrible!",
    "The new update looks okay, nothing special but it works.",
    "Amazing support team! They resolved my issue in less than 10 minutes. Highly recommend!",
    "Not sure if this feature is working correctly, getting some weird errors sometimes.",
    "Worst experience ever. The app crashed and I lost all my data. Completely unacceptable."
]

print("=" * 70)
print("   RAPPORT D'ANALYSE — Tickets support")
print("=" * 70)

for i, ticket in enumerate(tickets, 1):
    result = analyze_text(ticket)

    print(f"\n📧 Ticket #{i}")
    print(f"   Texte      : {ticket[:80]}...")
    print(f"   Prédiction : {result['label']} (confiance={result['confidence']:.3f})")
    print(f"   Scores     : {result['all_scores']}")
    print(f"   Top tokens : {[t for t, _ in result['highlighted_tokens'][:3]]}")
    print(f"   Routage    : {result['routing']}")
    print("-" * 70)

In [ ]:
# Visualisation du résultat d'analyse pour un ticket négatif
neg_ticket = tickets[0]
result = analyze_text(neg_ticket, top_k=8)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# --- Graphique 1 : Scores des 3 classes ---
bars = axes[0].bar(
    LABEL_NAMES,
    [result["all_scores"][n] for n in LABEL_NAMES],
    color=LABEL_COLORS, edgecolor="white", linewidth=1.5
)
axes[0].set_title("Scores softmax par classe", fontweight="bold")
axes[0].set_ylabel("Probabilité")
axes[0].set_ylim(0, 1)
for bar, val in zip(bars, [result["all_scores"][n] for n in LABEL_NAMES]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f"{val:.3f}", ha="center", fontweight="bold")
axes[0].grid(axis="y", alpha=0.4)

# --- Graphique 2 : Top tokens (attention) ---
tok_names = [t for t, _ in result["highlighted_tokens"]]
tok_scores = [s for _, s in result["highlighted_tokens"]]
norm_scores = np.array(tok_scores) / max(tok_scores)

colors_bars = ["#e74c3c" if s > 0.5 else "#3498db" for s in norm_scores]
axes[1].barh(tok_names[::-1], norm_scores[::-1], color=colors_bars[::-1], edgecolor="white")
axes[1].set_title("Tokens saillants (poids d'attention normalisés)", fontweight="bold")
axes[1].set_xlabel("Score d'attention (normalisé)")
axes[1].axvline(0.5, color="gray", linestyle="--", alpha=0.6)
axes[1].grid(axis="x", alpha=0.4)

red_patch   = mpatches.Patch(color='#e74c3c', label='Très saillant (>0.5)')
blue_patch  = mpatches.Patch(color='#3498db', label='Modérément saillant')
axes[1].legend(handles=[red_patch, blue_patch], fontsize=8)

plt.suptitle(f"Analyse : \"{neg_ticket[:60]}...\"\n→ {result['routing']}",
             fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig("analyze_text_result.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Visualisation sauvegardée : analyze_text_result.png")

---
## 🎯 Récapitulatif des livrables

| Livrable | Fichier / Variable | Description |
|----------|-------------------|-------------|
| Répartition des classes | `class_distribution.png` | Distribution train/val/test |
| Checkpoint DistilBERT | `./best_distilbert_sentiment/` | Modèle + tokenizer fine-tunés |
| Histogramme de confiance | `confidence_histograms.png` | Calibration sur le test set |
| Heatmaps d'attention | `attention_heatmaps.png` | Attention [CLS]→tokens par classe |
| Analyse de ticket | `analyze_text_result.png` | Exemple de sortie `analyze_text()` |
| Utilitaire d'inférence | `analyze_text()` | Fonction prête pour la production |

## 🚀 Prochaines étapes pour le déploiement

1. **API REST** : Encapsuler `analyze_text()` dans un endpoint FastAPI
2. **Seuil de confiance** : Affiner les seuils de routage selon les besoins métier
3. **Monitoring** : Enregistrer les prédictions en production pour détecter la dérive
4. **Multilinguisme** : Remplacer par `distilbert-base-multilingual-cased` pour les marchés non-anglophones
5. **Feedback loop** : Collecter les corrections humaines pour un re-fine-tuning périodique